# HDFS Task 1 — Baseline và mô hình đề xuất
**Chạy thử trước, chưa phải kết quả báo cáo cuối.** Notebook tự đọc dữ liệu đã giải nén bằng đường dẫn tương đối.

Đặt notebook cạnh thư mục `HDFS_v1`, hoặc ngay trong `HDFS_v1` cạnh `preprocessed`. Mở thư mục dự án bằng VS Code, chọn Python kernel và **Run All**.

Các mô hình: LR, LR balanced, DeepLog-style LSTM (normal-only), MLP count-only, GRU sequence-only, GRU + count, GRU + count weighted. DeepLog-style là bản triển khai cơ chế dự đoán event kế tiếp, có xử lý prefix ngắn; không phải tái hiện toàn bộ paper.

- `QUICK_RUN=True`: lấy mẫu phân tầng 20.000 block, tối đa 5 epoch, seed 42. Để kiểm tra pipeline.
- `QUICK_RUN=False`: toàn bộ dữ liệu, tối đa 30 epoch; có thể đặt `SEEDS=[42,43,44]` để chạy nhiều seed.
- Đây là **chế độ A trong tài liệu**: dùng Event_traces và ma trận tiền xử lý có sẵn. Chưa có raw-log parser chỉ fit trên train, API, hoặc demo.
- Test cũ đã được xem trong baseline trước; số mới vẫn là thực nghiệm phát triển, không bằng chứng đánh giá độc lập trên dữ liệu mới.


In [ ]:
# Chỉ bỏ dấu # nếu thiếu thư viện, sau đó Restart Kernel.
# %pip install numpy pandas scikit-learn matplotlib joblib torch tqdm
# Nếu muốn GPU NVIDIA, cài bản PyTorch phù hợp tại https://pytorch.org/get-started/locally/


## 1 Cấu hình và tìm data


In [ ]:
from pathlib import Path
import ast, re, json, random, time, copy, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, precision_score, recall_score,
    f1_score, accuracy_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay)
from tqdm.auto import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

QUICK_RUN = True
QUICK_BLOCKS = 20_000
SEEDS = [42]
SPLIT_SEED = 42
SPLIT_MODE = 'stratified'  # hoặc 'group_sequence': stress test không trùng chuỗi
BATCH_SIZE = 128
WINDOW_BATCH = 1024
CONTEXT = 10
EPOCHS = 5 if QUICK_RUN else 30
PATIENCE = 3 if QUICK_RUN else 5
LR_RATE = 1e-3
RECALL_TARGET = 0.99
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_MODELS = ['LR', 'LR_balanced', 'DeepLog_style', 'MLP_count',
              'GRU', 'GRU_count', 'GRU_count_weighted']

def find_data():
    roots = [Path.cwd(), Path.cwd() / 'HDFS_v1']
    roots += [p for p in Path.cwd().iterdir() if p.is_dir() and p.name != 'HDFS_v1']
    matches = []
    for root in roots:
        for folder in [root / 'preprocessed', root]:
            if all((folder / n).is_file() for n in
                   ['Event_traces.csv', 'Event_occurrence_matrix.csv', 'anomaly_label.csv']):
                if folder.resolve() not in matches: matches.append(folder.resolve())
    if len(matches) != 1:
        raise FileNotFoundError(f'Tìm thấy {len(matches)} thư mục dữ liệu: {matches}. '
            'Mở đúng thư mục chứa notebook và data đã giải nén trong VS Code. '
            f'Thư mục kernel hiện tại: {Path.cwd()}')
    return matches[0]

DATA = find_data()
OUT = Path.cwd() / 'hdfs_task1_output' / time.strftime('%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=False)
print('Data:', DATA, '\nOutput:', OUT, '\nDevice:', DEVICE)
print('QUICK RUN — không dùng làm kết quả cuối' if QUICK_RUN else 'FULL DATA')

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SPLIT_SEED)


## 2 Đọc và kiểm tra dữ liệu
Join bằng BlockId; chỉ count và event được dùng làm đầu vào. Không đưa Label, Type hoặc BlockId vào model. Giữ block khác nhau có cùng pattern. Đối chiếu số event trong chuỗi với vector đếm để tránh dùng sai mapping.


In [ ]:
matrix = pd.read_csv(DATA / 'Event_occurrence_matrix.csv')
labels = pd.read_csv(DATA / 'anomaly_label.csv')
traces = pd.read_csv(DATA / 'Event_traces.csv')
for name, frame in [('matrix', matrix), ('labels', labels), ('traces', traces)]:
    frame.columns = frame.columns.str.strip()
    if 'BlockId' not in frame: raise ValueError(f'{name}: thiếu BlockId; cột: {frame.columns.tolist()}')
    if frame.BlockId.isna().any() or frame.BlockId.duplicated().any():
        raise ValueError(f'{name}: BlockId thiếu hoặc trùng, cần kiểm tra dữ liệu nguồn.')
    frame['BlockId'] = frame.BlockId.astype(str)
if not set(labels.Label).issubset({'Normal', 'Anomaly'}):
    raise ValueError('Nhãn không hợp lệ hoặc thiếu.')
if not (set(matrix.BlockId) == set(labels.BlockId) == set(traces.BlockId)):
    raise ValueError('Ba file không có cùng tập BlockId; dừng để tránh mất mẫu âm thầm.')
FEATURES = sorted([c for c in matrix if re.fullmatch(r'E\d+', c)], key=lambda c:int(c[1:]))
if not FEATURES: raise ValueError('Không tìm thấy cột E1, E2, ...')
seq_candidates = [c for c in traces if c.lower() in
                  ['features','eventsequence','event_sequence','sequence','events','trace']]
if len(seq_candidates) != 1:
    raise ValueError(f'Không xác định duy nhất cột chuỗi: {traces.columns.tolist()}')
SEQ_COL = seq_candidates[0]

def parse_events(value):
    if pd.isna(value): raise ValueError('Chuỗi event bị thiếu')
    text = str(value).strip()
    # Cho phép [E1,E2], ['E1', 'E2'], [1, 2] hoặc chuỗi token cách nhau bằng dấu cách.
    tokens = re.findall(r'E\d+|(?<![A-Za-z_])\d+', text)
    residue = re.sub(r'E\d+|(?<![A-Za-z_])\d+', '', text)
    if not tokens or re.sub(r"[\s,;\[\]\(\)'\"]", '', residue):
        raise ValueError(f'Định dạng event không được hỗ trợ: {text[:100]}')
    return tuple(t if t.startswith('E') else 'E'+str(int(t)) for t in tokens)

frame = matrix[['BlockId'] + FEATURES].merge(labels[['BlockId','Label']], on='BlockId', validate='one_to_one')
frame = frame.merge(traces[['BlockId', SEQ_COL]], on='BlockId', validate='one_to_one')
counts = frame[FEATURES].to_numpy(dtype=np.float32)
if not np.isfinite(counts).all() or (counts < 0).any() or (counts != np.floor(counts)).any():
    raise ValueError('Count phải hữu hạn, nguyên và không âm.')
raw_sequences = [parse_events(x) for x in tqdm(frame[SEQ_COL], desc='Đọc chuỗi')]
event_col = {e:i for i,e in enumerate(FEATURES)}
for i, seq in enumerate(tqdm(raw_sequences, desc='Đối chiếu sequence/count')):
    if any(e not in event_col for e in seq): raise ValueError(f'Event chưa có cột đếm tại block {i}')
    observed = np.bincount([event_col[e] for e in seq], minlength=len(FEATURES))
    if not np.array_equal(observed, counts[i]):
        raise ValueError(f'Sequence/count không khớp tại {frame.BlockId.iloc[i]}. Kiểm tra mapping trước khi train.')
y = frame.Label.eq('Anomaly').to_numpy(dtype=np.int64)
block_ids = frame.BlockId.to_numpy()
audit = {'blocks':len(frame), 'features':len(FEATURES), 'normal':int((y==0).sum()),
         'anomaly':int(y.sum()), 'duplicate_count_vectors':int(frame[FEATURES].duplicated().sum())}
print(audit)
display(frame[['BlockId','Label']].head())


## 3 Chia tập và tạo biểu diễn
Lấy tập con phân tầng trước khi chia trong chế độ nhanh. Split theo block trước khi tạo cửa sổ. Vocabulary chỉ học từ train; PAD=0, UNK=1, BOS=2. Nhánh đếm dùng log1p + StandardScaler chỉ fit train; LR giữ StandardScaler trên count gốc như baseline cũ. Không cắt chuỗi.

`group_sequence` là stress test riêng, không đảm bảo tỷ lệ lớp 70/15/15 chính xác. Dừng nếu một tập thiếu lớp; không tìm seed đẹp theo kết quả test.


In [ ]:
all_idx = np.arange(len(y))
if QUICK_RUN and len(y) > QUICK_BLOCKS:
    selected, _ = train_test_split(all_idx, train_size=QUICK_BLOCKS, stratify=y, random_state=SPLIT_SEED)
    selected = np.sort(selected)
else: selected = all_idx
seq_hash = np.array([hashlib.sha256(','.join(s).encode()).hexdigest() for s in raw_sequences])
if SPLIT_MODE == 'stratified':
    tr, rest = train_test_split(selected, test_size=.30, stratify=y[selected], random_state=SPLIT_SEED)
    va, te = train_test_split(rest, test_size=.50, stratify=y[rest], random_state=SPLIT_SEED)
elif SPLIT_MODE == 'group_sequence':
    a,b = next(GroupShuffleSplit(n_splits=1, test_size=.30, random_state=SPLIT_SEED).split(selected, groups=seq_hash[selected]))
    tr, rest = selected[a], selected[b]
    a,b = next(GroupShuffleSplit(n_splits=1, test_size=.50, random_state=SPLIT_SEED).split(rest, groups=seq_hash[rest]))
    va, te = rest[a], rest[b]
else: raise ValueError('SPLIT_MODE không hợp lệ')
for idx in [tr,va,te]:
    if len(np.unique(y[idx])) != 2: raise ValueError('Một tập thiếu lớp; cần thiết kế lại group split.')
assert not (set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
manifest = pd.concat([pd.DataFrame({'BlockId':block_ids[idx], 'split':name, 'label':y[idx],
                                   'sequence_hash':seq_hash[idx]}) for name,idx in [('train',tr),('val',va),('test',te)]])
manifest.to_csv(OUT/'split_manifest.csv', index=False)
display(manifest.groupby(['split','label']).size().unstack(fill_value=0))
seen = set(seq_hash[tr])
print('Tỷ lệ test có chuỗi từng xuất hiện trong train:', np.mean([h in seen for h in seq_hash[te]]))
train_events = sorted(set(e for i in tr for e in raw_sequences[i]), key=lambda e:int(e[1:]))
vocab = {e:i+3 for i,e in enumerate(train_events)}
VOCAB_SIZE = len(vocab)+3
sequences = [torch.tensor([vocab.get(e,1) for e in s], dtype=torch.long) for s in raw_sequences]
scaler = StandardScaler().fit(np.log1p(counts[tr]))
X = scaler.transform(np.log1p(counts)).astype(np.float32)
joblib.dump({'scaler':scaler,'features':FEATURES,'transform':'log1p'}, OUT/'count_preprocessing.joblib')
(OUT/'vocabulary.json').write_text(json.dumps(vocab, indent=2), encoding='utf-8')


## 4 Hàm đánh giá và chọn ngưỡng bằng validation


In [ ]:
def measures(truth, scores, threshold):
    pred = scores >= threshold
    tn,fp,fn,tp = confusion_matrix(truth,pred,labels=[0,1]).ravel()
    return dict(Precision=precision_score(truth,pred,zero_division=0),
        Recall=recall_score(truth,pred,zero_division=0), F1=f1_score(truth,pred,zero_division=0),
        Accuracy=accuracy_score(truth,pred), AP=average_precision_score(truth,scores),
        TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp), FPR=fp/max(fp+tn,1))

def select_threshold(truth, scores, recall_target=None):
    p,r,t = precision_recall_curve(truth,scores)
    f = 2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
    if recall_target is None: j=int(np.argmax(f))
    else:
        candidates=np.flatnonzero(r[:-1]>=recall_target)
        j=int(candidates[np.argmax(p[:-1][candidates])])
    return float(t[j])

val_records=[]; trained={}; histories=[]
def register(name, seed, model, scores, seconds, protocol='supervised'):
    threshold=select_threshold(y[va],scores)
    recall_threshold=select_threshold(y[va],scores,RECALL_TARGET)
    row=dict(model=name,seed=seed,protocol=protocol,threshold=threshold,
             recall_threshold=recall_threshold,train_seconds=seconds,**measures(y[va],scores,threshold))
    val_records.append(row);trained[(name,seed)]=(model,row)
    pd.DataFrame(val_records).to_csv(OUT/'validation.csv',index=False)
    print(name,seed,'val F1:',round(row['F1'],4))


## 5 LR và LR balanced


In [ ]:
for name in ['LR','LR_balanced']:
    if name not in RUN_MODELS: continue
    start=time.perf_counter()
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,
        class_weight='balanced' if name.endswith('balanced') else None,random_state=SPLIT_SEED))
    model.fit(counts[tr],y[tr]);seconds=time.perf_counter()-start
    register(name,SPLIT_SEED,model,model.predict_proba(counts[va])[:,1],seconds)
    joblib.dump(model,OUT/f'{name}.joblib')


## 6 MLP count và GRU có giám sát


In [ ]:
class Blocks(Dataset):
    def __init__(self, indices): self.indices=np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self,k):
        i=self.indices[k]
        return sequences[i],torch.from_numpy(X[i]),float(y[i])

def collate_blocks(batch):
    seq,count,target=zip(*batch)
    return pad_sequence(seq,batch_first=True,padding_value=0),torch.tensor([len(s) for s in seq]),torch.stack(count),torch.tensor(target)

def block_loader(idx,shuffle=False):
    return DataLoader(Blocks(idx),batch_size=BATCH_SIZE,shuffle=shuffle,collate_fn=collate_blocks,num_workers=0)

class Classifier(nn.Module):
    def __init__(self,mode):
        super().__init__();self.mode=mode
        if mode!='count':
            self.embedding=nn.Embedding(VOCAB_SIZE,32,padding_idx=0)
            self.gru=nn.GRU(32,64,batch_first=True)
        if mode!='sequence': self.count_branch=nn.Sequential(nn.Linear(X.shape[1],32),nn.ReLU())
        width={'count':32,'sequence':64,'fusion':96}[mode]
        self.head=nn.Sequential(nn.Linear(width,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,1))
    def forward(self,seq,length,count):
        parts=[]
        if self.mode!='count':
            packed=pack_padded_sequence(self.embedding(seq),length.cpu(),batch_first=True,enforce_sorted=False)
            _,h=self.gru(packed);parts.append(h[-1])
        if self.mode!='sequence': parts.append(self.count_branch(count))
        return self.head(torch.cat(parts,dim=1)).squeeze(1)

@torch.no_grad()
def classifier_scores(model,idx):
    model.eval();result=[]
    for seq,length,count,target in block_loader(idx):
        result.append(model(seq.to(DEVICE),length,count.to(DEVICE)).sigmoid().cpu().numpy())
    return np.concatenate(result)

for seed in SEEDS:
    for name,mode,weighted in [('MLP_count','count',False),('GRU','sequence',False),
            ('GRU_count','fusion',False),('GRU_count_weighted','fusion',True)]:
        if name not in RUN_MODELS:continue
        seed_all(seed);model=Classifier(mode).to(DEVICE)
        weight=float((y[tr]==0).sum()/y[tr].sum()) if weighted else 1.0
        loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight,device=DEVICE))
        optimizer=torch.optim.Adam(model.parameters(),lr=LR_RATE)
        best=-1.;stale=0;start=time.perf_counter();best_state=None
        for epoch in range(1,EPOCHS+1):
            model.train();total=0.;n=0
            for seq,length,count,target in tqdm(block_loader(tr,True),desc=f'{name} {seed} epoch {epoch}',leave=False):
                optimizer.zero_grad(set_to_none=True)
                loss=loss_fn(model(seq.to(DEVICE),length,count.to(DEVICE)),target.to(DEVICE))
                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);optimizer.step()
                total+=loss.item()*len(target);n+=len(target)
            scores=classifier_scores(model,va);threshold=select_threshold(y[va],scores)
            f1=measures(y[va],scores,threshold)['F1']
            histories.append(dict(model=name,seed=seed,epoch=epoch,loss=total/n,val_F1=f1))
            print(f'{name} epoch {epoch}: loss={total/n:.4f}, val_F1={f1:.4f}')
            if f1>best+1e-6:best=f1;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
            else:stale+=1
            if stale>=PATIENCE:break
        seconds=time.perf_counter()-start;model.load_state_dict(best_state)
        register(name,seed,model,classifier_scores(model,va),seconds)
        torch.save(best_state,OUT/f'{name}_{seed}.pt')


## 7 DeepLog style — chỉ huấn luyện trên Normal
LSTM học dự đoán event kế tiếp từ tối đa 10 event trước. Prefix ngắn được giữ với độ dài thực; block một event dùng BOS. Đây là biến thể rõ ràng để không bỏ block ngắn, không phải kiến trúc chẩn đoán đầy đủ của paper.

Điểm block là hạng xấu nhất của event thật trong các dự đoán; UNK luôn có điểm cao nhất. Ngưỡng chọn bằng validation tương đương chọn top-k. Cùng test với model supervised nhưng khác lượng thông tin nhãn dùng train.


In [ ]:
class Windows(Dataset):
    def __init__(self,indices):
        self.indices=np.asarray(indices)
        self.sizes=np.array([max(len(sequences[i])-1,1) for i in self.indices])
        self.ends=np.cumsum(self.sizes)
    def __len__(self):return int(self.ends[-1])
    def __getitem__(self,k):
        owner=int(np.searchsorted(self.ends,k,side='right'))
        offset=k-(self.ends[owner-1] if owner else 0)
        seq=sequences[self.indices[owner]]
        if len(seq)==1:context=torch.tensor([2]);target=seq[0]
        else:
            pos=int(offset)+1;context=seq[max(0,pos-CONTEXT):pos];target=seq[pos]
        return context,target,owner

def collate_windows(batch):
    seq,target,owner=zip(*batch)
    return pad_sequence(seq,batch_first=True,padding_value=0),torch.tensor([len(s) for s in seq]),torch.stack(target),np.array(owner)

def window_loader(indices,shuffle=False):
    return DataLoader(Windows(indices),batch_size=WINDOW_BATCH,shuffle=shuffle,num_workers=0,collate_fn=collate_windows)

class NextEvent(nn.Module):
    def __init__(self):
        super().__init__();self.embedding=nn.Embedding(VOCAB_SIZE,32,padding_idx=0)
        self.lstm=nn.LSTM(32,64,num_layers=2,batch_first=True)
        self.head=nn.Linear(64,VOCAB_SIZE)
    def forward(self,seq,length):
        packed=pack_padded_sequence(self.embedding(seq),length.cpu(),batch_first=True,enforce_sorted=False)
        _,(h,_)=self.lstm(packed);logits=self.head(h[-1])
        # PAD/UNK/BOS không phải event đích học được.
        return logits.masked_fill(torch.arange(VOCAB_SIZE,device=logits.device)[None,:]<3,-1e9)

@torch.no_grad()
def next_scores(model,indices):
    model.eval();scores=np.zeros(len(indices),dtype=np.float32)
    for seq,length,target,owner in window_loader(indices):
        logits=model(seq.to(DEVICE),length);target=target.to(DEVICE)
        order=logits.argsort(dim=1,descending=True)
        ranks=(order==target[:,None]).to(torch.int64).argmax(dim=1)+1
        ranks[target==1]=VOCAB_SIZE
        np.maximum.at(scores,owner,ranks.cpu().numpy())
    return scores

if 'DeepLog_style' in RUN_MODELS:
    for seed in SEEDS:
        seed_all(seed);model=NextEvent().to(DEVICE)
        optimizer=torch.optim.Adam(model.parameters(),lr=LR_RATE)
        best=-1.;stale=0;best_state=None;start=time.perf_counter()
        normal_train=tr[y[tr]==0]
        for epoch in range(1,EPOCHS+1):
            model.train();total=0.;n=0
            for seq,length,target,_ in tqdm(window_loader(normal_train,True),desc=f'DeepLog {seed} epoch {epoch}',leave=False):
                optimizer.zero_grad(set_to_none=True)
                loss=nn.functional.cross_entropy(model(seq.to(DEVICE),length),target.to(DEVICE))
                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);optimizer.step()
                total+=loss.item()*len(target);n+=len(target)
            scores=next_scores(model,va);threshold=select_threshold(y[va],scores)
            f1=measures(y[va],scores,threshold)['F1']
            histories.append(dict(model='DeepLog_style',seed=seed,epoch=epoch,loss=total/n,val_F1=f1))
            print(f'DeepLog epoch {epoch}: loss={total/n:.4f}, val_F1={f1:.4f}')
            if f1>best+1e-6:best=f1;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
            else:stale+=1
            if stale>=PATIENCE:break
        seconds=time.perf_counter()-start;model.load_state_dict(best_state)
        register('DeepLog_style',seed,model,next_scores(model,va),seconds,'normal_only')
        torch.save(best_state,OUT/f'DeepLog_style_{seed}.pt')
pd.DataFrame(histories).to_csv(OUT/'training_history.csv',index=False)


## 8 Khóa lựa chọn rồi đánh giá test
Chọn cấu hình bằng **mean F1 validation** qua các seed; chưa xem test khi chọn. Báo riêng normal-only và supervised. AP của DeepLog dùng điểm hạng, không phải xác suất. Thời gian dưới đây là tổng scoring chia số block, gồm batching và chuyển dữ liệu; không phải p50/p95 của một API.


In [ ]:
validation=pd.DataFrame(val_records)
selection=validation.groupby(['protocol','model']).F1.mean().reset_index()
winners=selection.loc[selection.groupby('protocol').F1.idxmax()].reset_index(drop=True)
display(winners)
winners.to_csv(OUT/'selected_by_validation.csv',index=False)
results=[];predictions=[]
for (name,seed),(model,row) in trained.items():
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    start=time.perf_counter()
    if name.startswith('LR'):scores=model.predict_proba(counts[te])[:,1]
    elif name=='DeepLog_style':scores=next_scores(model,te)
    else:scores=classifier_scores(model,te)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    for operating,threshold in [('best_val_F1',row['threshold']),('val_recall99',row['recall_threshold'])]:
        results.append(dict(model=name,seed=seed,protocol=row['protocol'],operating=operating,
            threshold=threshold,mean_scoring_ms=1000*elapsed/len(te),**measures(y[te],scores,threshold)))
    predictions.append(pd.DataFrame({'BlockId':block_ids[te],'model':name,'seed':seed,
        'truth':y[te],'score':scores,'prediction':(scores>=row['threshold']).astype(int),
        'seen_sequence_in_train':[h in seen for h in seq_hash[te]],
        'length':[len(sequences[i]) for i in te],
        'unknown_events':[int((sequences[i]==1).sum()) for i in te]}))
result=pd.DataFrame(results);prediction=pd.concat(predictions,ignore_index=True)
result.to_csv(OUT/'test_metrics.csv',index=False)
prediction.to_csv(OUT/'test_predictions.csv',index=False)
display(result[result.operating=='best_val_F1'].round(4))
summary=result.groupby(['protocol','model','operating'])[['F1','Precision','Recall','AP']].agg(['mean','std'])
summary.to_csv(OUT/'test_summary.csv');display(summary.round(4))
print('std=NaN khi chỉ chạy một seed; không phải lỗi.')


## 9 Confusion matrix và phân tích lỗi


In [ ]:
for name in winners.model:
    sub=prediction[(prediction.model==name)&(prediction.seed==prediction[prediction.model==name].seed.min())]
    fig,ax=plt.subplots(figsize=(4,4))
    ConfusionMatrixDisplay.from_predictions(sub.truth,sub.prediction,display_labels=['Normal','Anomaly'],ax=ax,colorbar=False)
    ax.set_title(name+' — first seed');fig.tight_layout();fig.savefig(OUT/f'confusion_{name}.png',dpi=150);plt.show()
errors=prediction[prediction.truth!=prediction.prediction].copy()
errors['error']=np.where(errors.truth==0,'FP','FN')
errors.to_csv(OUT/'errors.csv',index=False)
display(errors.head(30))
pattern_rows=[]
for (name,seed,is_seen),sub in prediction.groupby(['model','seed','seen_sequence_in_train']):
    pattern_rows.append(dict(model=name,seed=seed,seen=is_seen,n=len(sub),
        anomaly_count=int(sub.truth.sum()),F1=f1_score(sub.truth,sub.prediction,zero_division=0),
        Recall=recall_score(sub.truth,sub.prediction,zero_division=0)))
pd.DataFrame(pattern_rows).to_csv(OUT/'seen_unseen.csv',index=False)
display(pd.DataFrame(pattern_rows))
print('Nhóm không có anomaly: F1/Recall=0 theo quy ước; không dùng để kết luận model kém.')


## 10 Lưu cấu hình và kiểm tra nạp model
Các file `.pt` chỉ chứa state_dict; cần chạy lại ô định nghĩa lớp và nạp preprocessing/vocabulary đúng bản. Chỉ nạp joblib từ nguồn tin cậy.


In [ ]:
config=dict(quick_run=QUICK_RUN,quick_blocks=QUICK_BLOCKS,seeds=SEEDS,split_seed=SPLIT_SEED,
    split_mode=SPLIT_MODE,epochs=EPOCHS,patience=PATIENCE,context=CONTEXT,
    batch_size=BATCH_SIZE,window_batch=WINDOW_BATCH,lr=LR_RATE,vocab_size=VOCAB_SIZE,
    embedding_dim=32,hidden_size=64,features=FEATURES,device=str(DEVICE),
    protocol='preprocessed_features_mode_A',audit=audit,
    versions={'python':platform.python_version(),'torch':torch.__version__,
              'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__})
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
for (name,seed),(model,row) in trained.items():
    if name.startswith('LR'):
        restored=joblib.load(OUT/f'{name}.joblib')
        assert np.allclose(model.predict_proba(counts[te[:3]]),restored.predict_proba(counts[te[:3]]))
    else:
        restored=NextEvent() if name=='DeepLog_style' else Classifier(model.mode)
        restored.load_state_dict(torch.load(OUT/f'{name}_{seed}.pt',map_location='cpu',weights_only=True))
        for key,value in model.state_dict().items():assert torch.equal(value.cpu(),restored.state_dict()[key])
print('Đã lưu và kiểm tra nạp checkpoint:',OUT)


## Cách đọc kết quả và bước tiếp theo
- Xem `validation.csv`, `selected_by_validation.csv` trước; `test_metrics.csv` chứa kết quả test ở hai chế độ ngưỡng.
- `LR_balanced`/`GRU_count_weighted` có thể tăng Recall nhưng giảm Precision. Không mặc định trọng số luôn tốt hơn.
- So sánh GRU với GRU_count để kiểm tra nhánh count; MLP_count với fusion để xem thông tin chuỗi có ích không.
- Chạy đầy đủ: đổi `QUICK_RUN=False`; đặt `SEEDS=[42,43,44]` nếu đủ thời gian; Restart & Run All. Không đổi lựa chọn theo test.
- Notebook chưa có parser raw-log chỉ fit train, bootstrap CI, đo p50/p95 hoặc ứng dụng/API. Đây là bản chạy thử chế độ A, không phải toàn bộ sản phẩm KLTN.

Nguồn phương pháp: [DeepLog](https://users.cs.utah.edu/~lifeifei/papers/deeplog.pdf), [LogAnomaly](https://www.ijcai.org/proceedings/2019/0658.pdf), [Loghub](https://github.com/logpai/loghub). Fusion là cấu hình thực nghiệm của nhóm, không tuyên bố thuật toán hoàn toàn mới.
